# Anonymization Agent

This notebook demonstrates the anonymization pipeline:
regex detection, LLM audit of residual PHI, pseudonymization, and
LLM verification -- all behind a single `anonymize()` call.

In [ ]:
from agentic_patterns.agents.anonymization import AnonymizationAgent, PrintAnonymizationListener

agent = AnonymizationAgent(listener=PrintAnonymizationListener(stream_events=True))

## Example 1: Full Pipeline

A synthetic discharge summary with multiple PHI categories: patient name, MRN, SSN, dates,
doctor name, phone, age >89, address, and facility name. The regex detector catches structured
patterns (MRN, SSN, phone, dates, ages); the LLM audit catches the rest (names, hospital name,
street address, city, state). Dates are epoch-shifted so the earliest date maps to
2000-01-01 and all others shift by the same offset, preserving intervals.

In [ ]:
NOTE = """\
DISCHARGE SUMMARY
Facility: Springfield General Hospital
Patient: Margaret Thompson
MRN: 4478-2291
SSN: 321-54-9876
Age: 94 y/o

Admission Date: January 10, 2025
Discharge Date: January 17, 2025

Attending: Dr. Rajesh Patel
Phone: (555) 867-5309

Patient Margaret Thompson, a 94 y/o female, was admitted on January 10, 2025
for acute exacerbation of COPD. She resides at 742 Evergreen Terrace, Springfield,
IL 62704. Her primary care physician, Dr. Patel, was notified upon admission.

Hospital course was uncomplicated. Discharged January 17, 2025 in stable condition
with follow-up scheduled for January 24, 2025.
"""

In [ ]:
result = await agent.anonymize(NOTE)
print(result.redacted_text)

In [ ]:
print(f"\n--- Regex detections: {len(result.detection_spans)} ---")
for span in result.detection_spans:
    print(f"  {span.label.value:25s} {NOTE[span.start:span.end]!r}")

print(f"\n--- LLM audit findings: {len(result.audit_spans)} ---")
for span in result.audit_spans:
    print(f"  {span.label.value:25s} {NOTE[span.start:span.end]!r}")

## Example 2: Pseudonym Consistency Across Documents

A second note for the same patient produces the same pseudonyms for matching entities
(same PATIENT, MRN, DOCTOR tokens), enabling cross-document linkage without revealing
real identifiers. The vault persists mappings to `vault.json`, so consistency holds
across separate runs too.

In [ ]:
SECOND_NOTE = """\
PROGRESS NOTE
Patient: Margaret Thompson
MRN: 4478-2291
Date: January 24, 2025

Follow-up visit. Patient reports improved breathing. Dr. Rajesh Patel
reviewed the latest spirometry results. Continue current medications.
"""

In [ ]:
result2 = await agent.anonymize(SECOND_NOTE)
print(result2.redacted_text)